# EX14 — Pandas Real-World Exercises

**What you'll learn:** loading & inspecting data, cleaning, filtering, groupby/aggregation,
merging, datetime handling, and a mini business-reporting case study.

See `Developer_Guide.md` and `User_Guide_BrainFriendly.md` in this folder before starting.


## 1. Load & Inspect
We'll build a small synthetic sales dataset (no external file needed) so this runs anywhere.

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(1)
n = 200
dates = pd.date_range("2024-01-01", periods=60, freq="D")
df = pd.DataFrame({
    "order_id": range(1, n+1),
    "date": np.random.choice(dates, n),
    "product": np.random.choice(["Coffee","Tea","Juice","Water","Soda"], n),
    "region": np.random.choice(["North","South","East","West"], n),
    "units": np.random.randint(1, 10, n),
    "unit_price": np.random.choice([2.5, 3.0, 1.5, 1.0, 2.0], n),
})
df.loc[df.sample(frac=0.05, random_state=2).index, "units"] = np.nan  # inject missing data
df["revenue"] = df["units"] * df["unit_price"]
df.head()


In [ ]:
df.info()
print(df.describe())
print("Missing values:\n", df.isna().sum())


### TODO 1
Drop rows with missing `units`, but keep a copy of the original `df` untouched (call the cleaned version `df_clean`).

In [ ]:
# TODO
df_clean = None
print(len(df), len(df_clean) if df_clean is not None else None)


<details><summary>Solution</summary>

```python
df_clean = df.dropna(subset=['units']).copy()
df_clean['revenue'] = df_clean['units'] * df_clean['unit_price']
```
</details>

## 2. Filtering & Querying
**Pointer:** `.query()` is often more readable than chained boolean masks for simple filters.

In [ ]:
high_value = df_clean[df_clean.revenue > 15]
print(len(high_value))

# equivalent with .query
high_value_q = df_clean.query("revenue > 15")
print(len(high_value_q))


### TODO 2
Find all orders from the 'North' region for the product 'Coffee' with more than 5 units.

In [ ]:
# TODO
result = None
print(len(result) if result is not None else None)


<details><summary>Solution</summary>

```python
result = df_clean.query("region == 'North' and product == 'Coffee' and units > 5")
```
</details>

## 3. GroupBy & Aggregation — Business Reporting
**Pointer:** this is the #1 real-world Pandas pattern — 'total X by Y'.

In [ ]:
report = df_clean.groupby("product").agg(
    total_units=("units", "sum"),
    total_revenue=("revenue", "sum"),
    avg_order_value=("revenue", "mean"),
).sort_values("total_revenue", ascending=False)
report


### TODO 3
Build a report of total revenue by `region` AND `product` (a 2-level groupby), then find which single (region, product) combo generated the most revenue.

In [ ]:
# TODO
region_product_report = None
top_combo = None
print(top_combo)


<details><summary>Solution</summary>

```python
region_product_report = df_clean.groupby(["region","product"])["revenue"].sum().sort_values(ascending=False)
top_combo = region_product_report.idxmax()
```
</details>


## 4. Merging Data Sources
Real-world data usually lives in more than one table. Let's join a `region -> manager` lookup table.

In [ ]:
region_managers = pd.DataFrame({
    "region": ["North","South","East","West"],
    "manager": ["Asha","Ben","Carla","Diego"],
})
merged = df_clean.merge(region_managers, on="region", how="left")
merged.head()


### TODO 4
Compute total revenue managed by each manager (i.e., sum of revenue for their region).

In [ ]:
# TODO
revenue_by_manager = None
print(revenue_by_manager)


<details><summary>Solution</summary>

```python
revenue_by_manager = merged.groupby('manager')['revenue'].sum().sort_values(ascending=False)
```
</details>

## 5. Datetime Handling & Resampling
**Pointer:** set the date as the index to unlock `.resample()`.

In [ ]:
ts = df_clean.set_index("date").sort_index()
daily_revenue = ts["revenue"].resample("D").sum()
weekly_revenue = ts["revenue"].resample("W").sum()
print(weekly_revenue.head())


### TODO 5
Find the single best week (highest total revenue) and the day-over-day % change in daily revenue (hint: `.pct_change()`).

In [ ]:
# TODO
best_week = None
daily_pct_change = None
print(best_week)


<details><summary>Solution</summary>

```python
best_week = weekly_revenue.idxmax()
daily_pct_change = daily_revenue.pct_change()
```
</details>

## Key Takeaways
- `groupby().agg()` with named aggregations is the cleanest way to build reports.
- `.merge(how=...)` for combining tables — pick the join type deliberately.
- Set a datetime column as index to unlock `.resample()` for time-series rollups.
- Always inspect (`info`, `describe`, `isna().sum()`) immediately after loading data.
- Use `.copy()` when creating a filtered DataFrame you intend to modify.
